<img src="https://github.com/ArchiColab/sitex/blob/main/images/banner.png?raw=1" width="1000"/>

# Green Space Accessibility — Clustering, Walking Time & Equity

**The question:** where can you sit under a tree within a short walk from home, and are you
allowed to? The objective of this notebook is to investigate who can walk to green space, how
evenly that access is spread across the ward, and where green exists but is not public.

This notebook measures how easily people can walk to green space, using two different green
layers:

- **Public green** comes from OpenStreetMap: parks, gardens, playgrounds, sports pitches and
  other places mapped for recreation. It shows where people are *allowed* to go.
- **Green environment** comes from the ESA WorldCover land-cover map: every patch of tree cover,
  shrubland or grassland of at least 1 ha. It shows where green *physically exists*, whether it
  is public, private or a plantation.

For each layer, the notebook calculates the walking time from every part of the ward to the
nearest green space and measures how evenly that access is spread (the Gini index). Comparing
the two layers shows where green is nearby but not public: places where opening up, connecting
or protecting land could make a real difference.

**Why two layers?** In Pleiku, OpenStreetMap maps very little green space (13 public places,
about 12 ha together, in September 2026), while WorldCover finds several hundred hectares of
tree cover and grassland in the same ward. Neither source alone tells the full story.

All the calculations live in `sitex.network.green_accessibility`; this notebook only sets
the parameters, calls the functions, and shows the results.

The notebook is optimized for Google Colab. Because it runs in the cloud, it can be used
on machines with limited computing power and in areas with limited bandwidth.

<img src="https://github.com/ArchiColab/sitex/blob/main/images/GreenAccess_overview.svg?raw=1" width="1000"/>

**Library:** [`sitex`](https://github.com/ArchiColab/sitex)

**Requires (run first):**
- `00-Data-Acquisition.ipynb`, Section 7: the ESA WorldCover land-cover map
  (`data/landcover/esa_worldcover_2021_{slug}.tif`).
- `00-Data-Acquisition.ipynb`, Section 9: the OSM walk network (`data/osm/{slug}_walk.gpkg`),
  the same network the isochrone and service-area notebooks use.
- `00-Data-Acquisition.ipynb`, Section 3b: the shared hex grid (`data/hex_grid_{slug}.gpkg`), for the
  export in Section 9.

**Output (`data/gdf/`):** the green layers and the H3 grid as GeoPackages (Section 8), and
`{slug}_hex_green.parquet` with its legend `{slug}_hex_green_legend.csv`, one row per hexagon of the
shared grid (Section 9).

Only the public green spaces are downloaded from OpenStreetMap while the notebook runs.

Walking speed is a constant 4.8 km/h on every street, the speed Transport for London uses
for walk times (Transport for London, 2015, *Assessing transport connectivity in London*,
Table 2.1). It is the same as in the isochrone and service-area notebooks (`03-NA03` to
`03-NA05`), so their walking times can be compared.

---
## Coordinate systems (CRS)

- **Analysis runs in the site's UTM zone** (metres; global UTM, not the local VN-2000 CRS). Buildings, streets and hexagons use it. Landsat and Sentinel-2 stay in UTM; Open Buildings tiles are brought into the site's UTM zone.; the terrain model, land cover and canopy height are saved in longitude/latitude (EPSG:4326) and are sampled or reprojected when a notebook needs metres. For initial analysis you can ignore VN-2000.
- **UTM is the default;** to combine with official VN-2000 data, reproject the vector data (see `00-Data-Acquisition`).
- **Sources of CRS:** the VN-2000 definitions are listed at [https://gis.vn/](https://gis.vn/kinh-tuyen-truc-ban-do-hanh-chinh-cap-tinh-thanh-viet-nam).

---
## 1. Install & import

<img src="https://github.com/ArchiColab/sitex/blob/main/images/GreenAccess_Setup.svg?raw=1" width="1000"/>


In [1]:
#@title <font color=#1B7192> Configuration </font>  { display-mode: "form" }

# One-time setup, if `sitex` isn't already installed in this kernel:
# %pip install "sitex[network] @ git+https://github.com/ArchiColab/sitex.git"

import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("Running in:", "Google Colab" if IN_COLAB else "Local environment")

if IN_COLAB:
    %pip install "sitex[network] @ git+https://github.com/ArchiColab/sitex.git"

    from google.colab import drive
    drive.mount("/content/gdrive")

    DATA_DIR = Path("/content/gdrive/MyDrive/Colab_Outputs")
    OUTPUT_DIR = Path("/content/gdrive/MyDrive/SiteX_Outputs")
else:
    DATA_DIR = Path("..") / "data"
    OUTPUT_DIR = Path("..") / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
from IPython.display import display

from sitex.core.config import CityConfig
from sitex.data import aoi as aoi_lib
from sitex.network import accessibility_common as ac
from sitex.network import green_accessibility as ga


Running in: Local environment


## 2. Configuration

| Setting | Default | Meaning |
|---|---|---|
| `WALK_TIME_MINUTES` | 15 | Walking time that counts as good access to green space. |
| `H3_RESOLUTION` | 9 | Size of the hexagon grid: each hexagon covers about 0.1 km² (roughly 330 m across). Keep 9: it is the resolution of the shared hex grid from `00-Data-Acquisition.ipynb`, which the export in Section 9 needs. |
| `MIN_PATCH_HA` | 1.0 | Smallest WorldCover green patch that counts. Smaller patches, such as single gardens or tree rows, are left out. |
| `EDGE_BUFFER_M` | 25 | Street junctions within this distance of a green space count as its entrances. |

In [2]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 2. Configuration ###", flush=True)

city = CityConfig(
    place_name="Phường Pleiku, Gia Lai, Vietnam",
    slug="pleiku",  # same SLUG as the other notebooks
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
)
GDF_DIR = city.data_dir / "gdf"
GDF_DIR.mkdir(parents=True, exist_ok=True)

WALK_TIME_MINUTES = 15   # walking time that counts as good access
H3_RESOLUTION = 9       # hexagons of about 0.1 km²
MIN_PATCH_HA = 1.0       # smallest WorldCover green patch that counts (ha)
EDGE_BUFFER_M = 25       # junctions this close to a green space count as its entrances (m)

WC_PATH = city.data_dir / "landcover" / f"esa_worldcover_2021_{city.slug}.tif"
if not WC_PATH.exists():
    raise FileNotFoundError(f"{WC_PATH} not found -- run 00-Data-Acquisition.ipynb Section 7 first.")

# The saved walk network -- the same one 03-NA03 to 03-NA05 use
WALK_PATH = city.data_dir / "osm" / f"{city.slug}_walk.gpkg"
if not WALK_PATH.exists():
    raise FileNotFoundError(f"{WALK_PATH} not found -- run 00-Data-Acquisition.ipynb Section 9 first.")

print(f"Local EPSG : {city.local_epsg}")
print(f"WorldCover : {WC_PATH}")
print(f"Walk network: {WALK_PATH}")

print(f"--- 2. done in {time.time() - _t0:.2f}s ---\n")


### 2. Configuration ###
Local EPSG : 32649
WorldCover : ..\data\landcover\esa_worldcover_2021_pleiku.tif
Walk network: ..\data\osm\pleiku_walk.gpkg
--- 2. done in 0.03s ---



---
## 3. Two green layers

The notebook downloads the OpenStreetMap green spaces and loads the saved walking network.
Of the green spaces, it keeps
only the places mapped for public recreation (`ga.PUBLIC_GREEN_LABELS`): parks, gardens,
nature reserves, playgrounds, sports pitches, commons, recreation grounds and village greens.
Farmland, orchards, plantations and water are left out, because they are green but not places
where people go to spend time.

The WorldCover map is turned into patches: neighbouring pixels of tree cover, shrubland or
grassland are joined into one patch, and patches smaller than `MIN_PATCH_HA` are dropped.
Hover over a patch on the map to see its area and its share of tree cover.

**Not all of the green environment is accessible.** A satellite cannot tell a park from a
coffee or rubber plantation, a private garden or a fenced compound. Read this layer as "green
that exists", not as "green that people may use".

The map shows both layers: public green in dark green, on top of the green environment in
light green.

In [3]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 3. Two green layers ###", flush=True)

print("Downloading green spaces from OpenStreetMap...")
green_gdf = ga.fetch_green_spaces(city.place_name)
public_gdf = ga.select_public_green(green_gdf)

# Saved walk network (both walking directions, 4.8 km/h) and the AOI boundary it was cut to
G = ac.load_walk_graph(WALK_PATH)
boundary_poly = aoi_lib.resolve_aoi_by_place(city.place_name).boundary
print(f"Walking network ({WALK_PATH.name}): {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

env_gdf = ga.worldcover_green_patches(WC_PATH, city.local_epsg, min_area_ha=MIN_PATCH_HA, boundary=boundary_poly)


def _hectares(gdf):
    return gdf.to_crs(epsg=city.local_epsg).area.sum() / 10_000


print(f"\nOSM green polygons, all types : {len(green_gdf):4,}  ({_hectares(green_gdf):7,.1f} ha)")
print(f"  of which public green       : {len(public_gdf):4,}  ({_hectares(public_gdf):7,.1f} ha)")
print(f"WorldCover patches >= {MIN_PATCH_HA} ha  : {len(env_gdf):4,}  ({_hectares(env_gdf):7,.0f} ha)")
print("\nOSM green by type (only the public types are used as public green):")
print(green_gdf["green_space"].value_counts().to_string())

print(f"--- 3. done in {time.time() - _t0:.2f}s ---\n")
ga.plot_green_layers(public_gdf, env_gdf, boundary_poly)


### 3. Two green layers ###
Walking network (pleiku_walk.gpkg): 2,734 nodes, 6,795 edges

OSM green polygons, all types :   18  (  185.0 ha)
  of which public green       :   13  (   11.8 ha)
WorldCover patches >= 1.0 ha  :   65  (    944 ha)

OSM green by type (only the public types are used as public green):
green_space
leisure:park          8
leisure:pitch         3
landuse:farmland      3
leisure:playground    2
leisure:stadium       1
natural:water         1
--- 3. done in 2.73s ---



---
## 4. Green space clustering

<img src="https://github.com/ArchiColab/sitex/blob/main/images/GreenAccess_cluster.svg?raw=1" width="1000"/>

The green-environment patches are sorted into three size groups with K-means clustering, based
on their area only. Cluster 0 holds the smallest patches and cluster 2 the largest. The table
shows how many patches each group has and their size range. The clusters are a separate way of
reading the green layer: the following sections do not use them.

In [4]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 4. Green space clustering ###", flush=True)

clustered_gdf = ga.cluster_green_spaces(env_gdf, city.local_epsg, num_clusters=3)
sizes = clustered_gdf.groupby("cluster")["area_m2"].agg(["count", "min", "max", "sum"])
sizes[["min", "max", "sum"]] /= 10_000
sizes.columns = ["patches", "smallest (ha)", "largest (ha)", "total (ha)"]
print(sizes.round(1).to_string())

print(f"--- 4. done in {time.time() - _t0:.2f}s ---\n")
ga.plot_green_clusters(clustered_gdf)


### 4. Green space clustering ###
         patches  smallest (ha)  largest (ha)  total (ha)
cluster                                                  
0             60            1.0          21.6       200.6
1              3           59.1          82.0       207.6
2              2          183.4         352.0       535.4
--- 4. done in 0.07s ---



---
## 5. Walking time to green (H3 grid)

<img src="https://github.com/ArchiColab/sitex/blob/main/images/GreenAccess_accessibility.svg?raw=1" width="1000"/>

For every hexagon, the notebook calculates the walking time to the nearest green space, once for
each layer, up to `WALK_TIME_MINUTES`. The walk ends at the *edge* of a green space: at any
street junction inside it or within `EDGE_BUFFER_M` of it. This matters for large patches,
whose centre can be several hundred metres from the nearest entrance. A green space without any
street nearby is reached at the junction closest to it.

The first map shows the walking time to public green, the second the walking time to the green
environment. Hexagons farther than `WALK_TIME_MINUTES` from any green space are left blank.

In [5]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 5. Walking time to green (H3 grid) ###", flush=True)

access_kwargs = dict(walk_time_minutes=WALK_TIME_MINUTES, h3_resolution=H3_RESOLUTION,
                     local_epsg=city.local_epsg, edge_buffer_m=EDGE_BUFFER_M)
grid_public = ga.compute_green_accessibility(G, boundary_poly, public_gdf, **access_kwargs)
grid_env = ga.compute_green_accessibility(G, boundary_poly, env_gdf, **access_kwargs)

print(f"H3 grid (res {H3_RESOLUTION}): {len(grid_public):,} hexes")
for label, grid in [("Public green", grid_public), ("Green environment", grid_env)]:
    share = 100 * (grid["access_time_min"] <= WALK_TIME_MINUTES).mean()
    print(f"  {label:18s}: {share:5.1f}% of hexes within {WALK_TIME_MINUTES} min")

print(f"--- 5. done in {time.time() - _t0:.2f}s ---\n")
display(ga.plot_green_accessibility(grid_public, public_gdf, legend_name="Walking time to public green (min)"))
ga.plot_green_accessibility(grid_env, env_gdf, legend_name="Walking time to green environment (min)")


### 5. Walking time to green (H3 grid) ###
H3 grid (res 9): 217 hexes
  Public green      :  21.7% of hexes within 15 min
  Green environment : 100.0% of hexes within 15 min
--- 5. done in 1.62s ---



---
## 6. Comparison & spatial equity (Gini index)

<img src="https://github.com/ArchiColab/sitex/blob/main/images/GreenAccess_equity.svg?raw=1" width="1000"/>

The table compares the two layers:

| Column | Meaning |
|---|---|
| `pct_hexes_within` | Share of hexagons within `WALK_TIME_MINUTES` of a green space. |
| `median_walk_min` | Median walking time, for the hexagons within reach. |
| `gini_walk_time` | Gini index of the walking times, for the hexagons within reach: 0 means everyone has the same walk, values towards 1 mean very unequal walks. |

How to read the Gini index:
- It only includes hexagons within reach, so always read it together with `pct_hexes_within`.
  A layer with poor coverage can still show a low Gini index.
- It is **not meaningful for the green environment** in a green city. When most hexagons lie
  right next to green (a walk of 0 minutes), the few longer walks make the index high, even
  though access is excellent. Use the coverage share for that layer instead.

The map shows the walking-time inequality for public green, the layer where equity matters
most.

In [6]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 6. Comparison & spatial equity (Gini index) ###", flush=True)

summary = pd.DataFrame([
    ga.summarise_green_access(grid_public, WALK_TIME_MINUTES, "Public green (OSM)"),
    ga.summarise_green_access(grid_env, WALK_TIME_MINUTES, "Green environment (WorldCover)"),
]).round(3)
display(summary)
gini_score = summary.loc[0, "gini_walk_time"]

print(f"--- 6. done in {time.time() - _t0:.2f}s ---\n")
ga.plot_green_inequality(grid_public, gini_score, legend_name="Walking time to public green (min)")


### 6. Comparison & spatial equity (Gini index) ###


,green_layer,pct_hexes_within,median_walk_min,gini_walk_time
0,Public green (OSM),21.659,6.909,0.314
1,Green environment (WorldCover),100.000,0.000,0.804


--- 6. done in 0.03s ---



---
## 7. Where is green nearby, but not public?

Each hexagon is sorted into one of four classes, based on what lies within `WALK_TIME_MINUTES`:

| Class | Meaning for planning |
|---|---|
| Public green and green environment nearby | Well served. |
| Green nearby, but no public green | **The opportunity.** Green exists within walking distance, but none of it is mapped as public. Find out what this green is (a plantation, a private garden, a closed compound or an unmapped park), and whether it could be opened up, connected or protected. |
| Public green only | Public green without larger green patches around it, typically small city parks in dense areas. |
| No green nearby | Neither public green nor green environment within reach. |

Each class is a separate layer on the map, and the percentage in each layer name shows its
share of the ward.

Some of the "green nearby, but no public green" area may simply be parks that nobody has mapped
in OpenStreetMap yet. Checking these places on the ground or in satellite imagery is part of the
exercise.

In [7]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 7. Where is green nearby, but not public? ###", flush=True)

gap_grid = ga.classify_green_gap(grid_public, grid_env, WALK_TIME_MINUTES)
shares = 100 * gap_grid["gap_class"].value_counts(normalize=True).reindex(list(ga.GAP_CLASSES)).fillna(0)
for key, pct in shares.items():
    print(f"  {ga.GAP_CLASSES[key][0]:45s} {pct:5.1f}% of hexes")

print(f"--- 7. done in {time.time() - _t0:.2f}s ---\n")
ga.plot_green_gap(gap_grid, public_gdf)


### 7. Where is green nearby, but not public? ###
  Public green and green environment nearby      21.7% of hexes
  Green nearby, but no public green              78.3% of hexes
  Public green only                               0.0% of hexes
  No green nearby                                 0.0% of hexes
--- 7. done in 0.02s ---



---
## 8. Export

<img src="https://github.com/ArchiColab/sitex/blob/main/images/GreenAccess_export.svg?raw=1" width="1000"/>

Saves the maps as HTML and the comparison table as CSV in `outputs/`. The green layers, the
clusters and the H3 grid (the walking time to both layers and the gap class of every hexagon)
are saved as GeoPackages in `data/gdf/`, in the local metric coordinate system.

In [8]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 8. Export ###", flush=True)

maps = {
    "green_layers.html": ga.plot_green_layers(public_gdf, env_gdf, boundary_poly),
    "green_clusters.html": ga.plot_green_clusters(clustered_gdf),
    "green_accessibility_public.html": ga.plot_green_accessibility(
        grid_public, public_gdf, legend_name="Walking time to public green (min)"),
    "green_accessibility_environment.html": ga.plot_green_accessibility(
        grid_env, env_gdf, legend_name="Walking time to green environment (min)"),
    "green_spatial_inequality.html": ga.plot_green_inequality(
        grid_public, gini_score, legend_name="Walking time to public green (min)"),
    "green_gap.html": ga.plot_green_gap(gap_grid, public_gdf),
}
for name, m in maps.items():
    m.save(str(city.output_dir / name))
summary.to_csv(city.output_dir / "green_accessibility_summary.csv", index=False)
print(f"Saved {len(maps)} maps and green_accessibility_summary.csv to {city.output_dir}")


def export_gdf(gdf, columns, filename):
    out = gdf[columns + ["geometry"]].copy().to_crs(epsg=city.local_epsg)
    out.to_file(GDF_DIR / filename, driver="GPKG")
    return out


export_gdf(green_gdf, ["green_space"], "green_spaces_raw.gpkg")
export_gdf(public_gdf, ["green_space"], "green_public_osm.gpkg")
export_gdf(env_gdf, ["area_m2", "tree_share"], "green_environment_worldcover.gpkg")
export_gdf(clustered_gdf, ["area_m2", "tree_share", "cluster"], "green_clusters.gpkg")
export_gdf(gap_grid, ["h3_id", "access_public_min", "access_environment_min", "gap_class"],
           "green_accessibility_grid.gpkg")

print(f"GeoPackages saved to {GDF_DIR}  (CRS: EPSG:{city.local_epsg})")

print(f"--- 8. done in {time.time() - _t0:.2f}s ---\n")


### 8. Export ###
Saved 6 maps and green_accessibility_summary.csv to ..\outputs
GeoPackages saved to ..\data\gdf  (CRS: EPSG:32649)
--- 8. done in 1.87s ---



---
## 9. Export for interpretation: one row per hexagon

Every analysis notebook ends with a table like this one: its results per hexagon of the shared
hex grid (`hex_grid_{slug}.gpkg`, built in `00-Data-Acquisition.ipynb` Section 3b), keyed by
`h3_id`. The tables from different notebooks can then be joined and read side by side. The
numbers are **per hexagon, not per resident**.

This notebook's grid is already the shared grid, because `H3_RESOLUTION` is 9 like the grid file.
If you changed it, the cell below stops and asks you to set it back and re-run Section 5.

The table has the walking time to public green and to the green environment (empty when neither
is reachable within `WALK_TIME_MINUTES`), the class from Section 7, and `snap_dist_m`, the
distance from the hexagon centre to the street junction the walk starts from. A large
`snap_dist_m` means the hexagon is far from any street.

The file `{slug}_hex_green.parquet` and its legend `{slug}_hex_green_legend.csv` are written to
`data/gdf`.


In [9]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 9. Export for interpretation: one row per hexagon ###", flush=True)

import geopandas as gpd
import h3

hex_grid_path = city.data_dir / f"hex_grid_{city.slug}.gpkg"
if not hex_grid_path.exists():
    raise FileNotFoundError(f"{hex_grid_path} not found -- run 00-Data-Acquisition.ipynb Section 3b first.")
hexes = gpd.read_file(hex_grid_path, layer="hex")
if set(hexes["h3_id"]) != set(gap_grid["h3_id"]):
    raise ValueError(f"The grid of this notebook (H3_RESOLUTION = {H3_RESOLUTION}) is not the shared hex grid "
                     f"(resolution {h3.get_resolution(hexes['h3_id'].iloc[0])}). Set H3_RESOLUTION to that "
                     "resolution in Section 2 and re-run from Section 5.")

access_cols = ["access_public_min", "access_environment_min"]
hex_green = hexes[["h3_id"]].merge(gap_grid[["h3_id"] + access_cols + ["gap_class"]], on="h3_id")
hex_green = hex_green.merge(grid_public[["h3_id", "snap_dist_m"]], on="h3_id")
for col in access_cols:   # beyond the cutoff the walking time is not known: leave it empty
    hex_green[col] = hex_green[col].where(hex_green[col] <= WALK_TIME_MINUTES)

hex_green_path = GDF_DIR / f"{city.slug}_hex_green.parquet"
hex_green.to_parquet(hex_green_path, index=False)

# --- legend: one dictionary writes the table below and the legend file ---
H3_RES = h3.get_resolution(hexes["h3_id"].iloc[0])
walk = f"saved walk network at 4.8 km/h; the walk ends at the edge of a green space (a junction inside it or within {EDGE_BUFFER_M} m)"
empty = f"Empty when no green space is within {WALK_TIME_MINUTES} min"
LEGEND_COLUMNS = ["table", "column", "unit", "definition", "source", "limits"]
LEGEND = [
    ("hex_green", "h3_id", "-", f"Key of the hexagon (H3 cell, resolution {H3_RES})", f"hex_grid_{city.slug}.gpkg", "-"),
    ("hex_green", "access_public_min", "min", f"Walking time from the hexagon's street junction to the nearest public green space (up to {WALK_TIME_MINUTES} min)",
     f"OpenStreetMap green spaces mapped for recreation (Section 3); {walk}",
     f"{empty}. Only what volunteers mapped: an unmapped park counts as missing"),
    ("hex_green", "access_environment_min", "min", f"Walking time to the nearest green-environment patch (up to {WALK_TIME_MINUTES} min)",
     f"ESA WorldCover 2021, tree cover, shrubland and grassland patches of at least {MIN_PATCH_HA} ha (Section 3); {walk}",
     f"{empty}. Includes land people cannot use (plantations, private gardens, fenced compounds); shows 2021"),
    ("hex_green", "gap_class", "class", "; ".join(f"{key} = {label}" for key, (label, _color) in ga.GAP_CLASSES.items()),
     f"the two walking times above, within {WALK_TIME_MINUTES} min (Section 7)",
     "environment_only is where green exists but none of it is mapped as public: to be checked on the ground"),
    ("hex_green", "snap_dist_m", "m", "Distance from the hexagon centre to the street junction the walk starts from",
     "saved walk network", "The walk from the centre to the junction is not counted. A large value: the hexagon is far from any street"),
]
legend = pd.DataFrame(LEGEND, columns=LEGEND_COLUMNS)
legend_path = GDF_DIR / f"{city.slug}_hex_green_legend.csv"
legend.to_csv(legend_path, index=False, encoding="utf-8")

print(f"Hexagons: {len(hex_green)}")
print(hex_green["gap_class"].value_counts().to_string())
print(f"Saved -> {hex_green_path}")
print(f"Saved -> {legend_path}")
display(hex_green.drop(columns="h3_id").describe().round(2))
display(legend.drop(columns="table"))

print(f"--- 9. done in {time.time() - _t0:.2f}s ---\n")


### 9. Export for interpretation: one row per hexagon ###
Hexagons: 217
gap_class
environment_only    170
both                 47
Saved -> ..\data\gdf\pleiku_hex_green.parquet
Saved -> ..\data\gdf\pleiku_hex_green_legend.csv


,access_public_min,access_environment_min,snap_dist_m
count,47.00,217.00,217.00
mean,6.77,0.76,87.64
std,3.76,1.47,82.78
min,0.00,0.00,6.00
25%,4.47,0.00,34.22
50%,6.91,0.00,61.26
75%,9.08,0.83,107.98
max,13.41,6.87,450.67


,column,unit,definition,source,limits
0,h3_id,-,"Key of the hexagon (H3 cell, resolution 9)",hex_grid_pleiku.gpkg,-
1,access_public_min,min,Walking time from the hexagon's street junctio...,OpenStreetMap green spaces mapped for recreati...,Empty when no green space is within 15 min. On...
2,access_environment_min,min,Walking time to the nearest green-environment ...,"ESA WorldCover 2021, tree cover, shrubland and...",Empty when no green space is within 15 min. In...
3,gap_class,class,both = Public green and green environment near...,"the two walking times above, within 15 min (Se...",environment_only is where green exists but non...
4,snap_dist_m,m,Distance from the hexagon centre to the street...,saved walk network,The walk from the centre to the junction is no...


--- 9. done in 0.08s ---



---
## 10. Field check: what is this green?

Section 7 sorts the ward into four classes. Before the field trip, pick 2–3 hexagons for each
row:

| Class | What the model predicts | What to check on the ground |
|---|---|---|
| Green nearby, but no public green | Green exists, but none of it is mapped as public | What is it: a coffee or rubber plantation, a private garden, a fenced compound, an unmapped park? If it is a park, it can be added to OpenStreetMap. |
| Public green only | A small public green space | Can people use it: shade, benches, lighting, opening hours, a fence? Who is there? |
| No green nearby | Neither public green nor green environment within reach | Where do people go instead: a pagoda or church yard, school grounds, a café, a street with trees? |

**What to survey.** At each place, record:

- a photo, and what the green is: use, owner if visible, fence or gate
- who uses it, and when: count people for two minutes, by age group (children, adults, elderly)
- shade, seating, lighting, and how you get in

*Example of a record:* "Hexagon E, 'green nearby, but no public green': about 3 ha of coffee
plantation behind the houses, private, no fence along the lane. 2 children playing at its edge.
Not a park, but a path along its edge could make it a green route."

**Two more questions to discuss:**

- **Is 15 minutes the right threshold?** WHO Europe suggests that people should live within
  about 300 m of a public green space of at least 0.5–1 ha, roughly a 4-minute walk (WHO
  Regional Office for Europe, 2016, *Urban green spaces and health*). Re-run Section 2 with
  `WALK_TIME_MINUTES = 5`: how much of the ward is still served?
- **Double burden.** In QGIS, put the "No green nearby" hexagons over the Heat Exposure Index
  from the UHI notebook. Where are both? Who lives there?

---
## Notes

- **OpenStreetMap maps little green space in Pleiku.** Public green only contains what volunteers
  have mapped, so an unmapped park counts as missing (see Section 7).
- **The green environment includes land that people cannot use,** such as plantations, private
  gardens and fenced compounds (see Section 3).
- **Farmland is counted in neither layer.** OSM farmland is left out of public green, and
  WorldCover cropland is not part of the green environment.
- **Walks end at the edge of a green space,** at a junction inside it or within `EDGE_BUFFER_M`,
  not at its centre.
- **The Gini index only includes hexagons within reach,** and it is not meaningful for the green
  environment layer (see Section 6).
- **WorldCover shows 2021.** Green that has been built over since then still appears as green.
- **The OpenStreetMap green spaces are downloaded again on every run,** so results can change
  slightly when OpenStreetMap is edited. The walking network is the saved one: it only changes
  when you re-run Section 9 of the data acquisition.
- **Walking speed is the same as in the isochrone and service-area notebooks** (constant
  4.8 km/h), so their results can be compared.

---
## When the model is wrong

Where your notes and the map disagree, write down why. Common reasons:

- **Unmapped parks.** OpenStreetMap is made by volunteers; a park nobody mapped counts as
  missing.
- **Green used without being public.** Pagoda yards, school grounds after hours or the edge of a
  plantation can work as public space.
- **Public green nobody uses:** no shade, locked gates, a fence around a lawn.
- **WorldCover shows 2021,** and green built over since then still appears.

Write one sentence per disagreement: *"The map says …, I found …, because …"*. Green that is used
but not public, or public but not used, is where design can make the biggest difference.

## Where to go next

| Next step | Open |
|---|---|
| Where is it hottest? Overlay with the green gaps | [`02-ENV-UHI_Analysis.ipynb`](02-ENV-UHI_Analysis.ipynb) |
| Test how far a proposed park reaches: its entrances as points | [`03-NA04-Simple_Isochrone.ipynb`](03-NA04-Simple_Isochrone.ipynb) |

**Into the design phase:** "green nearby, but not public" is the opportunity: open it up, connect
it with a path, or protect it. Where there is no green at all, and especially where it is also
hot, a new public green space is the strongest case.